# Segmentacion lineal por tramos

Este notebook carga mediciones, busca segmentos con ajuste lineal y genera ecuaciones por tramo.


In [ ]:
%pip install numpy scipy matplotlib pandas openpyxl termcolor


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import openpyxl
import pandas as pd

from scipy.stats import linregress
from termcolor import cprint


In [ ]:
def prioridadErr(secuencia):
    """Devuelve la mayor cantidad de 1 consecutivos en una secuencia."""
    max_consecutivos = 0
    consecutivos_actuales = 0

    for elemento in secuencia:
        if elemento == 1:
            consecutivos_actuales += 1
            max_consecutivos = max(max_consecutivos, consecutivos_actuales)
        else:
            consecutivos_actuales = 0

    return max_consecutivos


## Carga y configuracion


In [ ]:
workbook = openpyxl.load_workbook("datos2_CANOPEN.xlsx")
sheet = workbook.active

data = []
for row in range(1, sheet.max_row):
    row_values = [row]
    for col in sheet.iter_cols(1, sheet.max_column):
        row_values.append(col[row].value)
    data.append(row_values)

df = pd.DataFrame(data, columns=["Cantidad", "output", "input"])
x_datos = df["input"].tolist()
y_datos = df["output"].tolist()


In [ ]:
x_datos_ = np.copy(x_datos)
y_datos_ = np.copy(y_datos)

print(x_datos)


In [ ]:
errAbsMax = 0.01 * np.abs(np.max(y_datos)) / 100
errAbsMax_Comprobar = 0.2 * np.abs(np.max(y_datos)) / 100

print(errAbsMax)
print(errAbsMax_Comprobar)


## Busqueda de segmentos


In [ ]:
minPoint = 3  # Evalua al menos 4 puntos por segmento.
despValores = []

while len(x_datos_) > minPoint:
    if minPoint < 3:
        print("*** inserte mas puntos ***")
        break

    errAcumuladoSecuencia = []
    r_cuadrado = []

    for i in range(minPoint, len(x_datos_) + 1):
        pendiente, intercepto, r_valor, _, _ = linregress(x_datos_[0:i], y_datos_[0:i])

        errAcumuladoAbs = []
        for ii in range(i):
            y_pred = pendiente * x_datos_[ii] + intercepto
            y_err = np.abs(y_pred - y_datos_[ii])

            if y_err < errAbsMax:
                errAcumuladoAbs.append(1)
            else:
                errAcumuladoAbs.append(0)

        errAcumuladoSecuencia.append(errAcumuladoAbs)
        r_cuadrado.append(r_valor**2)

    mejor_secuencia = max(
        enumerate(errAcumuladoSecuencia),
        key=lambda item: prioridadErr(item[1]),
    )
    posicion, secuencia = mejor_secuencia

    posicion_r2 = np.argmax(r_cuadrado) + minPoint - 1
    posicion = posicion + minPoint - 1

    cprint(f"la secuencia es: {secuencia} --------", "black", "on_yellow", attrs=["blink"])
    print("secuencia esta en la posicion:", posicion, "----------------------------")
    print("el mejor r2 esta en la posicion:", posicion_r2, "----------------------------")

    posicionFinal = posicion_r2 if posicion_r2 < posicion else posicion
    despValores.append(posicionFinal)

    desplazamiento = despValores[-1]
    x_datos_ = np.roll(x_datos_, shift=-desplazamiento)[: len(x_datos_) - desplazamiento]
    y_datos_ = np.roll(y_datos_, shift=-desplazamiento)[: len(y_datos_) - desplazamiento]

    if 1 < len(x_datos_) <= minPoint:
        despValores.append(len(x_datos_))
        break


## Ajuste, graficas y salida


In [ ]:
x_Nuevo = []
y_Nuevo = []
m_val_ = []
b_val_ = []

for i in range(len(despValores)):
    if i == 0:
        despMin = 0
        despMax = despValores[i]
    else:
        despMin += despValores[i - 1]
        despMax += despValores[i]

    x_Nuevo.append([])
    y_Nuevo.append([])
    x_Nuevo[i].append(x_datos[despMin : despMax + 1])
    y_Nuevo[i].append(y_datos[despMin : despMax + 1])

    m_val, b_val, _, _, _ = linregress(
        x_datos[despMin : despMax + 1],
        y_datos[despMin : despMax + 1],
    )
    m_val_.append(m_val)
    b_val_.append(b_val)

fig, ax = plt.subplots()

for x, y in zip(x_Nuevo, y_Nuevo):
    ax.plot(x[0], y[0], marker="o", linestyle="-", markersize=8)

for m, b, i in zip(m_val_, b_val_, range(len(despValores))):
    x_values = np.linspace(x_Nuevo[i][0][0], x_Nuevo[i][0][-1], 2)
    y_values = m * x_values + b
    ax.plot(x_values, y_values, label=f"y = {m:.2f}x + {b:.2f}")

plt.grid(which="both", linestyle="--", linewidth=0.5)
plt.minorticks_on()
plt.grid(which="minor", linestyle=":", linewidth=0.5)
plt.legend(loc="center left", bbox_to_anchor=(1, 0.5))
ax.set_title("Longitud vs Voltaje")
ax.set_xlabel("Eje X")
ax.set_ylabel("Eje Y")
plt.show()


In [ ]:
fig, ax = plt.subplots()

for x, y in zip(x_Nuevo, y_Nuevo):
    ax.plot(x[0], y[0], marker="o", linestyle="-", markersize=8)

plt.grid(which="both", linestyle="--", linewidth=0.5)
plt.minorticks_on()
plt.grid(which="minor", linestyle=":", linewidth=0.5)
ax.set_title("Longitud vs Voltaje")
ax.set_xlabel("Eje X")
ax.set_ylabel("Eje Y")
plt.show()


In [ ]:
fig, ax = plt.subplots()

for m, b, i in zip(m_val_, b_val_, range(len(despValores))):
    x_values = np.linspace(x_Nuevo[i][0][0], x_Nuevo[i][0][-1], 2)
    y_values = m * x_values + b
    ax.plot(x_values, y_values, label=f"y = {m:.2f}x + {b:.2f}")

print(m_val_)
plt.grid(which="both", linestyle="--", linewidth=0.5)
plt.minorticks_on()
plt.grid(which="minor", linestyle=":", linewidth=0.5)
plt.legend(loc="center left", bbox_to_anchor=(1, 0.5))
ax.set_title("Longitud vs Voltaje")
ax.set_xlabel("Eje X")
ax.set_ylabel("Eje Y")
plt.show()


In [ ]:
fig, ax = plt.subplots()

pendiente_global = (y_datos[1] - y_datos[-1]) / (x_datos[1] - x_datos[-1])
intercepto_global = y_datos[1] - pendiente_global * x_datos[1]

for i in range(len(x_datos)):
    predicted_y = pendiente_global * x_datos[i] + intercepto_global
    e_values = y_datos[i] - predicted_y
    ax.plot(x_datos[i], e_values, marker="o", color="red")

plt.xlabel("X Data")
plt.ylabel("Residuos")
plt.title("Residuo del error vs regresion lineal")
plt.show()


In [ ]:
fig, ax = plt.subplots()
y1, y2 = errAbsMax_Comprobar, -errAbsMax_Comprobar

for m, b, i in zip(m_val_, b_val_, range(len(despValores))):
    x_values = np.copy(x_Nuevo[i][0][:])
    y_values_a = np.copy(y_Nuevo[i][0][:])
    y_values_p = m * x_values + b
    y_values = y_values_a - y_values_p

    for j in range(len(y_values)):
        if y_values_a[j] > 0:
            y_errValues = 100 * np.abs(y_values[j]) / np.abs(np.max(y_datos))
        else:
            y_errValues = -0.1

        if y_errValues > 0.1:
            ax.scatter(x_values[j], y_values[j], color="red")
            ax.text(
                x_values[j],
                y_values[j],
                f"({x_values[j]:.2f}, {y_values_a[j]:.2f})",
                color="black",
                fontsize=8,
                ha="left",
                va="bottom",
            )
            cprint(
                f"Input medido = {x_values[j]} |err = {y_errValues:0.2f}% error mayor a 0.1% | "
                f"error={y_values[j]:0.2f} mm | y= {y_values_a[j]} | Yp = {y_values_p[j]:0.3f} ",
                "black",
                "on_yellow",
            )

    ax.fill_between(x_values, y1, y2, color="blue", alpha=0.04)
    plt.plot(x_values, np.full_like(x_values, y1))
    plt.plot(x_values, np.full_like(x_values, y2))
    ax.plot(x_values, y_values)

ax.set_title("Err Absoluto Longitud vs Input")
ax.set_xlabel("Voltaje")
ax.set_ylabel("Err Absoluto Longitud")
plt.grid(which="both", linestyle="--", linewidth=0.5)
plt.show()


In [ ]:
vol1 = 1.57

for i in range(len(despValores)):
    if vol1 >= x_Nuevo[i][0][0] and vol1 < x_Nuevo[i][0][-1]:
        resol = m_val_[i] * vol1 + b_val_[i]
        print(f"{resol:.2f}")
        break


In [ ]:
condicional_1 = "IN_VALUE"
condicional_2 = "OUT_LONG"

for i in range(len(despValores)):
    if i == 0:
        cprint(
            f"IF {condicional_1} >={x_Nuevo[i][0][0]} AND {condicional_1} < {x_Nuevo[i][0][-1]} THEN",
            "black",
            "on_yellow",
            attrs=["blink"],
        )
        cprint(
            f"    {condicional_2} :=  {m_val_[i]:.7f}*{condicional_1} + {b_val_[i]:.7f} ;    //Recta {i+1}",
            "black",
            "on_yellow",
            attrs=["blink"],
        )
    elif 0 < i < len(despValores) - 1:
        cprint(
            f"ELSIF {condicional_1} >={x_Nuevo[i][0][0]} AND {condicional_1} < {x_Nuevo[i][0][-1]} THEN",
            "black",
            "on_yellow",
            attrs=["blink"],
        )
        cprint(
            f"    {condicional_2} :=  {m_val_[i]:.7f}*{condicional_1} + {b_val_[i]:.7f} ;    //Recta {i+1}",
            "black",
            "on_yellow",
            attrs=["blink"],
        )
    else:
        cprint(
            f"ELSIF {condicional_1} >={x_Nuevo[i][0][0]} AND {condicional_1} <110000 THEN",
            "black",
            "on_yellow",
            attrs=["blink"],
        )
        cprint(
            f"    {condicional_2} :=  {m_val_[i]:.7f}*{condicional_1} + {b_val_[i]:.7f} ;    //Recta {i+1}",
            "black",
            "on_yellow",
            attrs=["blink"],
        )
        cprint("ELSE", "black", "on_yellow", attrs=["blink"])
        cprint(f"    {condicional_2} :=  0.0000 ; ", "black", "on_yellow", attrs=["blink"])
        cprint("END_IF", "black", "on_yellow", attrs=["blink"])
